# AutoFeatureFE — OpenML Benchmark

Runs the agent on datasets from two curated OpenML benchmark suites:

| Suite | ID | Datasets | Task |
|---|---|---|---|
| [OpenML-CC18](https://www.openml.org/s/99) | 99 | 15 selected | Classification (AUC) |
| [OpenML-CTR23](https://www.openml.org/s/353) | 353 | 15 selected | Regression (RMSE) |

For each dataset the agent runs `N_ITERATIONS_PER_DATASET` improvement steps,
starting from a standard-scaling baseline.  Results are saved under `benchmark_results/`
and aggregated into a comparison table at the end.

> **Time estimate**: ~10 min/dataset × 30 datasets = ~5 hours with `N_ITER=10`.
> Reduce `N_ITER` for a quick test.

In [ ]:
%pip install -q xgboost scikit-learn pandas numpy matplotlib anthropic openai openml

## 1. Configuration

In [ ]:
import sys, json, shutil, importlib, time, traceback
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR))

# ── USER SETTINGS ─────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY        = "sk-ant-..."   # ← paste your key
# OPENAI_API_KEY         = "sk-..."       # ← or use OpenAI
LLM_PROVIDER             = "anthropic"   # "anthropic" | "openai"

N_ITER                   = 10    # agent iterations per dataset (reduce for quick tests)
TOPK                     = 3     # pool size per dataset
COMPLEXITY_ALPHA         = 0.005
INCLUDE_DATA_PROFILE     = True  # one-time LLM dataset analysis (recommended)
RESULTS_DIR              = REPO_DIR / "benchmark_results"
RESULTS_DIR.mkdir(exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

BASELINE_PIPELINE = {
    "description": "baseline — standard scaling only",
    "steps": [{"op": "scale", "method": "standard"}]
}
print(f"Results will be saved to: {RESULTS_DIR}")

## 2. Dataset lists

In [ ]:
# OpenML-CC18 — 15 classification datasets (suite 99)
CC18_TASKS = [
    [
        3,
        "kr-vs-kp"
    ],
    [
        6,
        "letter"
    ],
    [
        11,
        "balance-scale"
    ],
    [
        12,
        "mfeat-factors"
    ],
    [
        14,
        "mfeat-fourier"
    ],
    [
        15,
        "breast-w"
    ],
    [
        16,
        "mfeat-karhunen"
    ],
    [
        18,
        "mfeat-morphological"
    ],
    [
        22,
        "mfeat-zernike"
    ],
    [
        23,
        "cmc"
    ],
    [
        28,
        "optdigits"
    ],
    [
        29,
        "credit-approval"
    ],
    [
        31,
        "credit-g"
    ],
    [
        32,
        "pendigits"
    ],
    [
        37,
        "diabetes"
    ]
]

# OpenML-CTR23 — 15 regression datasets (suite 353)
CTR23_TASKS = [
    [
        361234,
        "abalone"
    ],
    [
        361235,
        "airfoil-self-noise"
    ],
    [
        361236,
        "auction-verification"
    ],
    [
        361237,
        "concrete-strength"
    ],
    [
        361241,
        "physiochemical-protein"
    ],
    [
        361242,
        "superconductivity"
    ],
    [
        361243,
        "geographical-music"
    ],
    [
        361244,
        "solar-flare"
    ],
    [
        361247,
        "naval-propulsion"
    ],
    [
        361249,
        "white-wine"
    ],
    [
        361250,
        "red-wine"
    ],
    [
        361251,
        "grid-stability"
    ],
    [
        361252,
        "video-transcoding"
    ],
    [
        361253,
        "wave-energy"
    ],
    [
        361254,
        "sarcos"
    ]
]

print(f"CC18 : {len(CC18_TASKS)} datasets")
print(f"CTR23: {len(CTR23_TASKS)} datasets")
print(f"Total: {len(CC18_TASKS)+len(CTR23_TASKS)} datasets × {N_ITER} iterations")

## 3. Runner function

In [ ]:
def reset_state(task_cfg: dict):
    """Write task.json, reset pipeline.json, clear per-run state files."""
    (REPO_DIR / 'task.json').write_text(json.dumps(task_cfg, indent=2))
    (REPO_DIR / 'pipeline.json').write_text(json.dumps(BASELINE_PIPELINE, indent=2))
    for f in ['results.tsv', 'topk_pool.json']:
        p = REPO_DIR / f
        if p.exists(): p.unlink()


def run_dataset(task_id: int, name: str, task: str, metric: str) -> dict:
    """
    Run the agent on one OpenML task.  Returns a result dict.
    Saves per-dataset artefacts to RESULTS_DIR/<suite>/<name>/.
    """
    suite = "cc18" if task == "classification" else "ctr23"
    out   = RESULTS_DIR / suite / name
    out.mkdir(parents=True, exist_ok=True)

    task_cfg = {
        "task": task, "dataset": "openml",
        "openml_id": task_id, "metric": metric,
    }

    print(f"  → Resetting state...")
    reset_state(task_cfg)

    # Reload modules so stale cached state doesn't bleed between runs
    import prepare, field_types, operations, agent
    for m in [field_types, operations, prepare, agent]:
        importlib.reload(m)

    t0 = time.time()
    crashed = False
    try:
        agent.run(
            n_iterations         = N_ITER,
            topk                 = TOPK,
            complexity_alpha     = COMPLEXITY_ALPHA,
            include_history      = True,
            history_size         = 10,
            include_data_profile = INCLUDE_DATA_PROFILE,
            allow_freestyle      = False,
            anthropic_api_key    = ANTHROPIC_API_KEY if LLM_PROVIDER == "anthropic" else None,
            openai_api_key       = globals().get("OPENAI_API_KEY"),
            llm_provider         = LLM_PROVIDER,
            working_dir          = str(REPO_DIR),
        )
    except Exception:
        crashed = True
        traceback.print_exc()

    elapsed = time.time() - t0

    # Archive artefacts
    for fname in ['results.tsv', 'topk_pool.json', 'pipeline.json', 'task.json']:
        src = REPO_DIR / fname
        if src.exists():
            shutil.copy(src, out / fname)

    # Parse results
    results_file = out / 'results.tsv'
    if crashed or not results_file.exists():
        return {"task_id": task_id, "name": name, "task": task, "metric": metric,
                "crashed": True, "baseline": None, "best": None, "improvement_pct": None,
                "n_experiments": 0, "elapsed_s": elapsed}

    df_res = pd.read_csv(results_file, sep='\t')
    higher  = metric == "auc"
    base    = df_res.iloc[0]['val_score']
    best    = df_res['val_score'].max() if higher else df_res['val_score'].min()
    delta   = (best - base) / abs(base) * 100 if base != 0 else 0.0

    return {
        "task_id": task_id, "name": name, "task": task, "metric": metric,
        "crashed": False,
        "baseline": round(base, 6),
        "best":     round(best, 6),
        "improvement_pct": round(delta, 2),
        "n_experiments": len(df_res),
        "elapsed_s": round(elapsed, 1),
    }


print("Runner ready.")

## 4. Run CC18 (classification)

In [ ]:
cc18_results = []

for i, (task_id, name) in enumerate(CC18_TASKS):
    print(f"\n{'='*60}")
    print(f"[CC18  {i+1}/{len(CC18_TASKS)}]  {name}  (task {task_id})")
    print(f"{'='*60}")
    r = run_dataset(task_id, name, task="classification", metric="auc")
    cc18_results.append(r)
    status = "CRASHED" if r['crashed'] else f"AUC {r['baseline']:.4f} → {r['best']:.4f}  ({r['improvement_pct']:+.1f}%)"
    print(f"  Done: {status}  [{r['elapsed_s']:.0f}s]")

cc18_df = pd.DataFrame(cc18_results)
print("\nCC18 summary:")
cc18_df[['name','baseline','best','improvement_pct','n_experiments','elapsed_s','crashed']]

## 5. Run CTR23 (regression)

In [ ]:
ctr23_results = []

for i, (task_id, name) in enumerate(CTR23_TASKS):
    print(f"\n{'='*60}")
    print(f"[CTR23 {i+1}/{len(CTR23_TASKS)}]  {name}  (task {task_id})")
    print(f"{'='*60}")
    r = run_dataset(task_id, name, task="regression", metric="rmse")
    ctr23_results.append(r)
    status = "CRASHED" if r['crashed'] else f"RMSE {r['baseline']:.4f} → {r['best']:.4f}  ({r['improvement_pct']:+.1f}%)"
    print(f"  Done: {status}  [{r['elapsed_s']:.0f}s]")

ctr23_df = pd.DataFrame(ctr23_results)
print("\nCTR23 summary:")
ctr23_df[['name','baseline','best','improvement_pct','n_experiments','elapsed_s','crashed']]

## 6. Aggregate & export

In [ ]:
all_results = pd.concat([cc18_df, ctr23_df], ignore_index=True)

# Save combined summary
summary_path = RESULTS_DIR / 'benchmark_summary.csv'
all_results.to_csv(summary_path, index=False)
print(f"Saved summary → {summary_path}")

# Stats
for suite, df in [("CC18 (AUC, higher=better)", cc18_df), ("CTR23 (RMSE, lower=better)", ctr23_df)]:
    ok = df[~df['crashed']]
    improved = ok[ok['improvement_pct'].abs() > 0.1]
    print(f"\n{suite}")
    print(f"  Completed : {len(ok)}/{len(df)}")
    print(f"  Improved  : {len(improved)}/{len(ok)}")
    if len(ok):
        print(f"  Avg Δ     : {ok['improvement_pct'].mean():+.2f}%")
        print(f"  Max Δ     : {ok['improvement_pct'].abs().max():.2f}%  ({ok.loc[ok['improvement_pct'].abs().idxmax(),'name']})")

## 7. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

for ax, df, suite, metric, higher in [
    (axes[0], cc18_df,  "OpenML-CC18",  "AUC",  True),
    (axes[1], ctr23_df, "OpenML-CTR23", "RMSE", False),
]:
    ok = df[~df['crashed']].copy()
    ok = ok.sort_values('improvement_pct', ascending=False)

    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in ok['improvement_pct']]
    bars = ax.barh(ok['name'], ok['improvement_pct'], color=colors, alpha=0.85, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel(f'Improvement (%) in {metric}  ({"+" if higher else "-"} = better)')
    ax.set_title(f'{suite} — Agent improvement over baseline  (N={N_ITER} iters / dataset)',
                 fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())

    for bar, row in zip(bars, ok.itertuples()):
        x = bar.get_width()
        ax.text(x + 0.1, bar.get_y() + bar.get_height()/2,
                f"{row.best:.4f}", va='center', ha='left', fontsize=8)

plt.tight_layout(pad=3)
plt.savefig(RESULTS_DIR / 'benchmark_improvement.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved → {RESULTS_DIR}/benchmark_improvement.png")

In [ ]:
# Scatter: baseline vs best for each dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, suite, metric, higher in [
    (axes[0], cc18_df,  "CC18",  "AUC",  True),
    (axes[1], ctr23_df, "CTR23", "RMSE", False),
]:
    ok = df[~df['crashed']]
    ax.scatter(ok['baseline'], ok['best'], alpha=0.8, s=60, zorder=3)
    lims = [min(ok['baseline'].min(), ok['best'].min()),
            max(ok['baseline'].max(), ok['best'].max())]
    ax.plot(lims, lims, 'k--', lw=0.8, label='No change')
    for _, row in ok.iterrows():
        ax.annotate(row['name'], (row['baseline'], row['best']),
                    fontsize=7, alpha=0.7,
                    xytext=(4, 4), textcoords='offset points')
    ax.set(xlabel=f'Baseline {metric}', ylabel=f'Best {metric}',
           title=f'{suite} — Baseline vs Best')
    ax.legend()

plt.suptitle('AutoFeatureFE Benchmark Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'benchmark_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Per-dataset drill-down

Reload individual dataset results from `benchmark_results/` for deeper analysis.

In [ ]:
def load_dataset_results(name: str, suite: str = "cc18") -> pd.DataFrame:
    path = RESULTS_DIR / suite / name / 'results.tsv'
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep='\t')

def load_best_pipeline(name: str, suite: str = "cc18") -> dict:
    path = RESULTS_DIR / suite / name / 'pipeline.json'
    return json.loads(path.read_text()) if path.exists() else {}

# Example: inspect the most-improved CC18 dataset
ok_cc18 = cc18_df[~cc18_df['crashed']]
if len(ok_cc18):
    best_name = ok_cc18.loc[ok_cc18['improvement_pct'].idxmax(), 'name']
    df_run = load_dataset_results(best_name, "cc18")
    pipe   = load_best_pipeline(best_name, "cc18")
    print(f"Most-improved CC18 dataset: {best_name}")
    print(f"Pipeline steps: {[s['op'] for s in pipe.get('steps',[])]}")
    print(f"Description: {pipe.get('description','')}")
    df_run[['experiment','description','val_score','n_features','kept']].head(10)

## Notes

- **Resuming a partial run**: if the notebook was interrupted, cells 4 and 5
  will re-run completed datasets (fast, since `results.tsv` already exists)
  and continue from where it stopped.
- **Adding more datasets**: extend `CC18_TASKS` or `CTR23_TASKS` with any
  `(task_id, name)` tuple from `openml.study.get_suite(99).tasks`.
- **Per-dataset artefacts**: each dataset's `results.tsv`, `topk_pool.json`,
  and `pipeline.json` are saved under `benchmark_results/<suite>/<name>/`.
- **Changing iterations**: re-run a single dataset with more iterations:
  `run_dataset(task_id, name, task="classification", metric="auc")`